### ML analysis from data collected from model_processing_v3 notebook

In [1]:
#import statements
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import nilearn.image
import nilearn.plotting
import copy
from torch.utils.data import random_split
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
from pathlib import Path
import ants
import pydicom
import nibabel as nib
import os
from glob import glob
from tqdm import tqdm
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from torchinfo import summary
import nilearn

In [ ]:
# --- Load parquets + metadata, align rows in paired order --------------------
DATA = Path("model_data/adni")
meta = pd.read_csv(DATA / "paired_df_long.csv")

parquet_paths = {
    "t1_gm":  DATA / "t1_long_data"  / "t1_long_masked_gm.parquet",
    "t1_wm":  DATA / "t1_long_data"  / "t1_long_masked_wm.parquet",
    "t1_csf":  DATA / "t1_long_data"  / "t1_long_masked_csf.parquet",
    "dti_gm": DATA / "dti_long_data" / "dti_long_masked_gm_md.parquet",
    "dti_wm": DATA / "dti_long_data" / "dti_long_masked_wm_md.parquet",
    "dti_csf": DATA / "dti_long_data" / "dti_long_masked_csf_md.parquet",
}
key_col = {"t1_gm": "t1_image_subject_id", "t1_wm": "t1_image_subject_id", "t1_csf": "t1_image_subject_id",
           "dti_gm": "dti_image_subject_id", "dti_wm": "dti_image_subject_id", "dti_csf": "dti_image_subject_id"}

X_raw = {k: pd.read_parquet(p).loc[meta[key_col[k]]].values.astype(np.float32)
         for k, p in parquet_paths.items()}
for k, X in X_raw.items():
    print(f"{k:>7s}: {X.shape}")
print(f"meta: {meta.shape}  | groups: {meta['group'].value_counts(dropna=False).to_dict()}")

In [6]:
# --- StandardScaler + PCA(100) for each of the 4 datasets --------------------
N_PC = 100
scalers, pcas, X_pca = {}, {}, {}
for k, X in X_raw.items():
    sc = StandardScaler().fit(X)
    Xs = sc.transform(X)
    pca = PCA(n_components=N_PC, random_state=0).fit(Xs)
    scalers[k], pcas[k], X_pca[k] = sc, pca, pca.transform(Xs).astype(np.float32)
    print(f"{k:>7s}: {X.shape} -> {X_pca[k].shape}  | var explained = {pca.explained_variance_ratio_.sum():.3f}")

X_combined = np.hstack([X_pca[k] for k in ("t1_gm", "t1_wm", "dti_gm", "dti_wm")])
print(f"combined: {X_combined.shape}")

  t1_gm: (787, 451681) -> (787, 100)  | var explained = 0.631
  t1_wm: (787, 283320) -> (787, 100)  | var explained = 0.745
 dti_gm: (787, 451681) -> (787, 100)  | var explained = 0.850
 dti_wm: (787, 283320) -> (787, 100)  | var explained = 0.889
combined: (787, 400)


In [7]:
# --- SVM eval helper: balanced class weights, report bAcc / Sens / Spec ------
def eval_svm(X, y, name="", test_size=0.2, seed=0):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=test_size, stratify=y, random_state=seed)
    clf = SVC(class_weight="balanced", random_state=seed).fit(Xtr, ytr)
    yp = clf.predict(Xte)
    tn, fp, fn, tp = confusion_matrix(yte, yp, labels=[0, 1]).ravel()
    bacc = balanced_accuracy_score(yte, yp)
    sens = tp / (tp + fn) if (tp + fn) else float("nan")
    spec = tn / (tn + fp) if (tn + fp) else float("nan")
    print(f"  {name:>10s}  bAcc={bacc:.3f}  Sens={sens:.3f}  Spec={spec:.3f}  "
          f"(n_train={len(ytr)}, n_test={len(yte)}, pos%={y.mean():.2f})")
    return {"name": name, "bAcc": bacc, "Sens": sens, "Spec": spec}

def run_all(label_col):
    mask = meta[label_col].notna().values
    y = meta.loc[mask, label_col].astype(int).values
    print(f"\n=== {label_col}  (n={mask.sum()}, pos={y.sum()}, neg={(y==0).sum()}) ===")
    results = []
    for k in ("t1_gm", "t1_wm", "dti_gm", "dti_wm"):
        results.append(eval_svm(X_pca[k][mask], y, name=k))
    results.append(eval_svm(X_combined[mask], y, name="combined"))
    return pd.DataFrame(results)

In [8]:
# --- Binary classification: diag_label  (0 = CN, 1 = MCI/Dementia) -----------
diag_results = run_all("diag_label")
diag_results


=== diag_label  (n=742, pos=359, neg=383) ===
       t1_gm  bAcc=0.776  Sens=0.708  Spec=0.844  (n_train=593, n_test=149, pos%=0.48)
       t1_wm  bAcc=0.698  Sens=0.708  Spec=0.688  (n_train=593, n_test=149, pos%=0.48)
      dti_gm  bAcc=0.697  Sens=0.653  Spec=0.740  (n_train=593, n_test=149, pos%=0.48)
      dti_wm  bAcc=0.730  Sens=0.694  Spec=0.766  (n_train=593, n_test=149, pos%=0.48)
    combined  bAcc=0.746  Sens=0.764  Spec=0.727  (n_train=593, n_test=149, pos%=0.48)


,name,bAcc,Sens,Spec
0,t1_gm,0.776245,0.708333,0.844156
1,t1_wm,0.698323,0.708333,0.688312
2,dti_gm,0.696519,0.652778,0.740260
3,dti_wm,0.730339,0.694444,0.766234
4,combined,0.745581,0.763889,0.727273


In [9]:
# --- Binary classification: amyloid_label  (0 = A-, 1 = A+) ------------------
amy_results = run_all("amyloid_label")
amy_results


=== amyloid_label  (n=724, pos=376, neg=348) ===
       t1_gm  bAcc=0.728  Sens=0.827  Spec=0.629  (n_train=579, n_test=145, pos%=0.52)
       t1_wm  bAcc=0.659  Sens=0.747  Spec=0.571  (n_train=579, n_test=145, pos%=0.52)
      dti_gm  bAcc=0.627  Sens=0.640  Spec=0.614  (n_train=579, n_test=145, pos%=0.52)
      dti_wm  bAcc=0.644  Sens=0.573  Spec=0.714  (n_train=579, n_test=145, pos%=0.52)
    combined  bAcc=0.680  Sens=0.760  Spec=0.600  (n_train=579, n_test=145, pos%=0.52)


,name,bAcc,Sens,Spec
0,t1_gm,0.727619,0.826667,0.628571
1,t1_wm,0.659048,0.746667,0.571429
2,dti_gm,0.627143,0.640000,0.614286
3,dti_wm,0.643810,0.573333,0.714286
4,combined,0.680000,0.760000,0.600000
